# Week 2  
There are no specific questions to answer in your Jupyter Notebook files in this course; your general goal is to analyze your data, using the methods you have learned about in this course and in this program, and draw interesting conclusions. 

For Week 2, include concepts such as linear regression with lasso, ridge, and elastic net regression. This homework will be submitted for peer review and feedback in Week 3 in the assignment titled 3.4 Peer Review: Week 2 Jupyter Notebook. Complete your Jupyter Notebook homework by 11:59 pm ET on Sunday.

#### Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

#### Load Data

This analysis will focus on one dataset for the purpose of demonstration of key concepts learned.

In [2]:
diabetes_indicators = pd.read_csv('Modified_Datasets/diabetes_indicators_2.csv')

In [3]:
cervical_cancer = pd.read_csv('Modified_Datasets/cervical_cancer_2.csv')

#### Inspect Data

**Diabetes**

In [4]:
diabetes_indicators.info()

<class 'pandas.DataFrame'>
RangeIndex: 229781 entries, 0 to 229780
Data columns (total 23 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   Unnamed: 0            229781 non-null  int64
 1   Diabetes_012          229781 non-null  int64
 2   HighBP                229781 non-null  int64
 3   HighChol              229781 non-null  int64
 4   CholCheck             229781 non-null  int64
 5   BMI                   229781 non-null  int64
 6   Smoker                229781 non-null  int64
 7   Stroke                229781 non-null  int64
 8   HeartDiseaseorAttack  229781 non-null  int64
 9   PhysActivity          229781 non-null  int64
 10  Fruits                229781 non-null  int64
 11  Veggies               229781 non-null  int64
 12  HvyAlcoholConsump     229781 non-null  int64
 13  AnyHealthcare         229781 non-null  int64
 14  NoDocbcCost           229781 non-null  int64
 15  GenHlth               229781 non-null  int64


In [5]:
diabetes_indicators.head()

,Unnamed: 0,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0,0,1,1,1,40,1,0,0,0,...,1,0,5,18,15,1,0,9,4,3
1,1,0,0,0,0,25,1,0,0,1,...,0,1,3,0,0,0,0,7,6,1
2,2,0,1,1,1,28,0,0,0,0,...,1,1,5,30,30,1,0,9,4,8
3,3,0,1,0,1,27,0,0,0,1,...,1,0,2,0,0,0,0,11,3,6
4,4,0,1,1,1,24,0,0,0,1,...,1,0,2,3,0,0,0,11,5,4


In [6]:
Xd = diabetes_indicators.drop('Diabetes_012', axis = 1)
yd = diabetes_indicators['Diabetes_012']
Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, yd, test_size = 0.2, random_state = 42)

**Cervical Cancer**

In [7]:
cervical_cancer.info()

<class 'pandas.DataFrame'>
RangeIndex: 650 entries, 0 to 649
Data columns (total 29 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Unnamed: 0                          650 non-null    int64  
 1   Age                                 650 non-null    int64  
 2   Number of sexual partners           650 non-null    float64
 3   First sexual intercourse            650 non-null    float64
 4   Num of pregnancies                  650 non-null    float64
 5   Smokes                              650 non-null    float64
 6   Smokes (years)                      650 non-null    float64
 7   Smokes (packs/year)                 650 non-null    float64
 8   Hormonal Contraceptives             650 non-null    float64
 9   Hormonal Contraceptives (years)     650 non-null    float64
 10  IUD                                 650 non-null    float64
 11  IUD (years)                         650 non-null    floa

In [8]:
cervical_cancer.head()

,Unnamed: 0,Age,Number of sexual partners,First sexual intercourse,Num of pregnancies,Smokes,Smokes (years),Smokes (packs/year),Hormonal Contraceptives,Hormonal Contraceptives (years),...,STDs:Hepatitis B,STDs:HPV,Dx:CIN,Dx:HPV,Dx,Hinselmann,Schiller,Citology,Biopsy,Dx:Cancer
0,0,18,4.0,15.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0,0,0,0,0,0,0
1,1,15,1.0,14.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0,0,0,0,0,0,0
2,3,52,5.0,16.0,4.0,1.0,37.0,37.0,1.0,3.0,...,0.0,0.0,0,1,0,0,0,0,0,1
3,4,46,3.0,21.0,4.0,0.0,0.0,0.0,1.0,15.0,...,0.0,0.0,0,0,0,0,0,0,0,0
4,5,42,3.0,23.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0,0,0,0,0,0,0,0


In [9]:
Xc = cervical_cancer.drop('Dx:Cancer', axis = 1)
yc = cervical_cancer['Dx:Cancer']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size = 0.2, random_state = 42)

#### Lasso Regression

**Diabetes Indicators**

In [10]:
# Iterate over several alpha values
scores = []

for alpha in [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]:
    lasso = Lasso(alpha = alpha)
    lasso.fit(Xd_train, yd_train)
    yd_pred = lasso.predict(Xd_test)
    scores.append(mean_absolute_error(yd_test, yd_pred))
    
# MAE
print(scores)

[0.4736321592281506, 0.4735211606159276, 0.476494317158958, 0.4891083743819674, 0.5350250949736661, 0.5366465925203353, 0.5366535814030957, 0.5367076918628932]


**Cervical Cancer**

In [11]:
# Iterate over several alpha values
scores = []

for alpha in [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]:
    lasso = Lasso(alpha = alpha)
    lasso.fit(Xc_train, yc_train)
    yc_pred = lasso.predict(Xc_test)
    scores.append(mean_absolute_error(yc_test, yc_pred))
    
# MAE
print(scores)

[0.017927681544265144, 0.01794010868699649, 0.03507223200027815, 0.0580888078349785, 0.05867338242806305, 0.05976331360946745, 0.05976331360946745, 0.05976331360946745]


#### Ridge Regression

**Diabetes Indicators**

In [12]:
# Scale data
scaler = StandardScaler()

Xd_train_scaled = scaler.fit_transform(Xd_train)
Xd_test_scaled = scaler.transform(Xd_test)

In [13]:
# Instantiate model, iterate over alpha values
scores = []

for alpha in [0.0001, 0.1, 1.0, 10.0, 100.0, 1000.0, 1250.0, 1500.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(Xd_train_scaled, yd_train)
    yd_pred = ridge.predict(Xd_test_scaled)
    scores.append(mean_absolute_error(yd_test, yd_pred))

print(scores)

[0.4736564780171372, 0.47365646491033975, 0.47365634683194063, 0.4736551661328642, 0.4736433736518152, 0.4735271249122277, 0.4734953451255699, 0.4734637756611584]


**Cervical Cancer**

In [14]:
# Scale data
scaler = StandardScaler()

Xc_train_scaled = scaler.fit_transform(Xc_train)
Xc_test_scaled = scaler.transform(Xc_test)

In [15]:
# Instantiate model, iterate over alpha values
scores = []

for alpha in [0.0001, 0.1, 1.0, 10.0, 100.0, 1000.0, 1250.0, 1500.0]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(Xc_train_scaled, yc_train)
    yc_pred = ridge.predict(Xc_test_scaled)
    scores.append(mean_absolute_error(yc_test, yc_pred))

print(scores)

[0.01829379799596698, 0.018290495740589857, 0.01826197536434921, 0.018137938338937613, 0.02097400596647253, 0.037879509689893356, 0.04021266584857699, 0.04206877343155802]


#### Elastic Net Regression

**Diabetes Indicators**

In [16]:
# Features already scaled

In [17]:
# Instantiate model, search over multiple values of l1_ratio and alpha
cv_model = ElasticNetCV(
    l1_ratio=[0.0001, 0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
    alphas=np.logspace(-5, 1, 100),
    cv=5,
    random_state=42
)

In [18]:
cv_model.fit(Xd_train_scaled, yd_train)

,"l1_ratio l1_ratio: float or list of float, default=0.5Float between 0 and 1 passed to ElasticNet (scaling betweenl1 and l2 penalties). For ``l1_ratio = 0``the penalty is an L2 penalty. For ``l1_ratio = 1`` it is an L1 penalty.For ``0 < l1_ratio < 1``, the penalty is a combination of L1 and L2This parameter can be a list, in which case the differentvalues are tested by cross-validation and the one giving the bestprediction score is used. Note that a good choice of list ofvalues for l1_ratio is often to put more values close to 1(i.e. Lasso) and less close to 0 (i.e. Ridge), as in ``[.1, .5, .7,.9, .95, .99, 1]``.","[0.0001, 0.1, ...]"
,"alphas alphas: array-like or int, default=100Values of alphas to test along the regularization path, used for each l1_ratio.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`.",array([1.0000...00000000e+01])
,"cv cv: int, cross-validation generator or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- int, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For int/None inputs, :class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: 'auto', bool or array-like of shape (n_features, n_features), default='auto'Whether to use a precomputed Gram matrix to speed upcalculations. If set to ``'auto'`` let us decide. The Grammatrix can also be passed as argument.",'auto'
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``.",0.0001
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"verbose verbose: bool or int, default=0Amount of verbosity.",0


In [19]:
# Best alpha
cv_model.alpha_

np.float64(0.0008697490026177834)

In [20]:
# Best l1 ratio
cv_model.l1_ratio_

np.float64(0.1)

In [21]:
yd_pred = cv_model.predict(Xd_test_scaled)

In [22]:
mean_absolute_error(yd_test, yd_pred)

0.47360394783092674

Similar performance across all 3 models.

**Cervical Cancer**

In [23]:
# Instantiate model, search over multiple values of l1_ratio and alpha
cv_model = ElasticNetCV(
    l1_ratio=[0.0001, 0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
    alphas=np.logspace(-5, 1, 100),
    cv=5,
    random_state=42
)

In [24]:
cv_model.fit(Xc_train_scaled, yc_train)

,"l1_ratio l1_ratio: float or list of float, default=0.5Float between 0 and 1 passed to ElasticNet (scaling betweenl1 and l2 penalties). For ``l1_ratio = 0``the penalty is an L2 penalty. For ``l1_ratio = 1`` it is an L1 penalty.For ``0 < l1_ratio < 1``, the penalty is a combination of L1 and L2This parameter can be a list, in which case the differentvalues are tested by cross-validation and the one giving the bestprediction score is used. Note that a good choice of list ofvalues for l1_ratio is often to put more values close to 1(i.e. Lasso) and less close to 0 (i.e. Ridge), as in ``[.1, .5, .7,.9, .95, .99, 1]``.","[0.0001, 0.1, ...]"
,"alphas alphas: array-like or int, default=100Values of alphas to test along the regularization path, used for each l1_ratio.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`.",array([1.0000...00000000e+01])
,"cv cv: int, cross-validation generator or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- int, to specify the number of folds,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For int/None inputs, :class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: 'auto', bool or array-like of shape (n_features, n_features), default='auto'Whether to use a precomputed Gram matrix to speed upcalculations. If set to ``'auto'`` let us decide. The Grammatrix can also be passed as argument.",'auto'
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``.",0.0001
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"verbose verbose: bool or int, default=0Amount of verbosity.",0


In [25]:
# Best alpha
cv_model.alpha_

np.float64(0.1519911082952933)

In [26]:
# Best l1 ratio
cv_model.l1_ratio_

np.float64(0.0001)

In [27]:
yc_pred = cv_model.predict(Xc_test_scaled)

In [28]:
mean_absolute_error(yc_test, yc_pred)

0.02018272343417238

Lasso performed the best.